# Lesson14: LangChainの主要モジュール5【Agents（AIエージェント）】

## 事前準備
シークレット機能でOpenAI APIのAPIキーを設定し、以下2つのコードを実行しましょう。

In [1]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [2]:
!pip install langchain==0.3.26 openai==1.91.0 langchain-community==0.3.26 httpx==0.28.1 pydantic==2.11.7 chromadb==1.0.13

## 記述用

In [3]:
!pip install ddgs==9.10.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 73.9 MB/s eta 0:00:00


In [4]:
from langchain.agents import AgentType,initialize_agent
#LLMにどのツールを・どのような順序で使うかを律的に判断させる仕組み(AgentType:AIエージェントの種類を決めるための設定,initialize_agent:LLM、使用するツール（Tools）、エージェントのタイプなどを渡して、実際に動くエージェントを生成・初期化する)
from langchain.chat_models import ChatOpenAI
#PythonのAI開発ライブラリ「LangChain」で、OpenAI社が提供するChatGPTなどのチャットモデル（GPT-4など）を呼び出すためのコード
from langchain.tools import Tool
#PythonのフレームワークLangChainにおいて、LLM（大規模言語モデル）やAgent（エージェント）に利用させるカスタムツールを作成・定義するためのクラス
from ddgs import DDGS
#検索エンジン DuckDuckGo の検索結果（Web検索、画像、ニュース、動画など）をプログラムから取得・スクレイピングするため

#DuckDuckGo の検索 API ライブラリ（duckduckgo_search）を使用して Web 検索を行い、上位 3 件の検索結果を取得する関数
def search_web(query):
  results = DDGS().text(query,max_results=3)
  return results

#LangChainでカスタムツール（Tool）を作成する(やってることは既存の関数をLangchainのエージェントが理解できるツールに変換する（中身はツールが呼び出される関数、名前、説明文）)
search_tool = Tool.from_function(
    func=search_web,
    name="search_tool",
    description="最新のニュースなど、ネット上の情報を調べる時に使います。"
)

tools = [search_tool]
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

#LLMフレームワーク LangChain を使って「自立して思考・ツール実行するAIエージェント（ReActエージェント）」を構築・実行する
#①AIエージェントを初期化し、生成されたエージェントオブジェクトを agent_executor という変数に代入
#②エージェントの「頭脳」となる言語モデル（LLM）を指定
#③エージェントが利用できる「ツール（道具）のリスト」を指定
#④エージェントの思考アルゴリズム・挙動タイプ（フレームワーク）を指定（ZEROSHOTで考えREACT（思考→実行→観察のサイクルを回して答えを探す）→説明文をLLMが読み取ってこの質問はどのツールを使うか自分で判断）
#⑤コンソールログに詳細表示させる
agent_executor = initialize_agent(
    llm=llm,
    tools=tools,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)


agent_executor.run("2024年12月時点の日本の総理大臣は誰ですか？")

/tmp/ipykernel_1426/2665404456.py:23: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)
/tmp/ipykernel_1426/2665404456.py:31: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrat



> Entering new AgentExecutor chain...


2024年12月時点の日本の総理大臣について調べる必要があります。最新の情報を得るために、検索ツールを使用します。  
Action: search_tool  
Action Input: "2024年12月 日本 総理大臣"  
Observation: [{'title': '第2次石破内閣 - Wikipedia', 'href': 'https://ja.wikipedia.org/wiki/第2次石破内閣', 'body': 'May 21, 2026 - 第2次石破内閣（だいにじ いしばないかく）は、衆議院議員・自由民主党総裁の石破茂が第103代内閣総理大臣に任命され、2024年（令和6年）11月11日から2025年 （令和7年）10月21日まで続いた日本の内閣。 自由民主党と公...'}, {'title': '第1次石破内閣 - Wikipedia', 'href': 'https://ja.wikipedia.org/wiki/第1次石破内閣', 'body': 'May 17, 2026 - 第1次石破内閣（だいいちじ いしばないかく）は、衆議院議員・自由民主党総裁の石破茂が第102代内閣総理大臣に任命され、2024年（令和6年）10月1日から2024年（令和6年）11月11日まで続いた日本の内閣。 自由民主党と公...'}, {'title': '内閣総理大臣の一覧 - Wikipedia', 'href': 'https://ja.wikipedia.org/wiki/内閣総理大臣の一覧', 'body': 'July 8, 2026 - 2019年（令和元年）11月2...命中の総理大臣経験者は全員平成以降に就任した人物のみとなった。また、2025年（令和7年）10月17日に村山富市が死去したことで、全員が昭和生まれとなった。 · 2025年（令和7年）12月7日現在...'}]
Thought:2024年12月時点の日本の総理大臣は石破茂であることが分かりました。彼は2024年11月11日から2025年10月21日まで第103代内閣総理大臣を務めています。

Final Answer: 石破茂

> Finished chain.


'石破茂'

In [5]:
!pip install wikipedia==1.4.0

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=cfb407b38076036fdb5c4b5640421e4b8273d02ecb56ca21a9569226cc03149a
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


標準Tool

In [6]:
#大規模言語モデル（LLM）に外部ツールやAPIを使わせ、自律的に思考・行動・検証のループを行わせるAIエージェント機能や関数（create_agent など）を Python で読み込むため
#①エージェントの動作タイプ（どうやって思考してツールを使うかのルール）を定義する定数群です。
#② LLM とツール（今回は Wikipedia）をガッチャンコして、実際に動作するエージェントを作成する関数です。
#③Wikipedia や計算機など、エージェントが使える外部ツールを簡単に準備・読み込むための便利関数です。
from langchain.agents import AgentType,initialize_agent,load_tools
#PythonのAI開発ライブラリ「LangChain」で、OpenAI社が提供するChatGPTなどのチャットモデル（GPT-4など）を呼び出すためのコード
from langchain.chat_models import ChatOpenAI
#Python から Wikipedia の記事を検索・取得するための外部ライブラリ
import wikipedia

#Wikipediaサーバーに対して「誰が（どんなアプリが）アクセスしているか」を伝える設定
wikipedia.set_user_agent("langchain-app/0.1 (test@example.com)")
#】 検索やデータ取得の対象とするWikipediaの「言語」を日本語に設定
wikipedia.set_lang("ja")

#LLMの設定
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)
#Wikipediaの検索・参照ツール（API）を読み込む
tools = load_tools(["wikipedia"])

#LLMフレームワーク LangChain を使って「自立して思考・ツール実行するAIエージェント（ReActエージェント）」を構築・実行する
#①AIエージェントを初期化し、生成されたエージェントオブジェクトを agent_executor という変数に代入
#②エージェントの「頭脳」となる言語モデル（LLM）を指定
#③エージェントが利用できる「ツール（道具）のリスト」を指定
#④エージェントの思考アルゴリズム・挙動タイプ（フレームワーク）を指定（ZEROSHOTで考えREACT（思考→実行→観察のサイクルを回して答えを探す）→説明文をLLMが読み取ってこの質問はどのツールを使うか自分で判断）
#⑤コンソールログに詳細表示させる
agent_executor = initialize_agent(
    llm=llm,
    tools=tools,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent_executor.run("江戸時代の食文化について教えて")




> Entering new AgentExecutor chain...
江戸時代の食文化についての情報を得るために、Wikipediaを検索する必要があります。  
Action: wikipedia  
Action Input: 江戸時代 食文化  
Observation: Page: List of oldest companies
Summary: The oldest companies in the world are the brands and companies which remain operating (either in whole or in part) since inception, excluding associations and educational, government, or religious organizations.

Page: Kakigōri
Summary: Kakigōri (かき氷) is a Japanese shaved ice dessert flavored with syrup and a sweetener, often condensed milk.

Page: Shinise
Summary: Shinise (老舗, lit. 'old shop(s)') are Japanese businesses that have been in operation for at least a hundred years, some more than a thousand years. Japan has significantly more very old businesses than any other country, and  Shinise products enjoy a particular prestige in Japan.
Thought:Observation: The search did not return relevant information about the food culture of the Edo period. I need to refine my search.  
Action: wikipedia  
Action

'江戸時代の食文化は、主に米を中心とした食事が特徴でした。江戸時代には、魚や野菜を使った料理が多く、特に新鮮な魚介類が人気でした。寿司や刺身、天ぷらなどの料理が発展し、また、味噌や醤油といった調味料も重要な役割を果たしていました。さらに、食事は季節感を大切にし、旬の食材を取り入れることが重視されていました。江戸の町人文化の影響で、屋台や食堂も多く、一般市民が手軽に美味しい食事を楽しむことができる環境が整っていました。'

In [7]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

tools = load_tools(["llm-math"], llm=llm)

agent_executor = initialize_agent(
    llm=llm,
    tools=tools,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent_executor.run("320の15%は何ですか？")



> Entering new AgentExecutor chain...
320の15%を計算する必要があります。これは320に0.15を掛けることと同じです。  
Action: Calculator  
Action Input: 320 * 0.15  
Observation: Answer: 48.0
Thought:I now know the final answer.  
Final Answer: 48.0

> Finished chain.


'48.0'

In [8]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

tools = load_tools(["llm-math"], llm=llm)

agent_executor = initialize_agent(
    llm=llm,
    tools=tools,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

query = """
ある都市の人口は現在100万人です。毎年2%ずつ人口が増えると仮定しますが、
5年後に新たな法律が施行され、その影響で毎年3%ずつ人口が減ると予想されています。
この都市の人口が10年後にどれだけ変化しているか計算してください。
"""

agent_executor.run(query)



> Entering new AgentExecutor chain...
まず、最初の5年間の人口の増加を計算します。次に、次の5年間で人口がどのように減少するかを計算します。この2つの段階を経て、10年後の人口を求めます。

Action: Calculator
Action Input: 1000000 * (1 + 0.02) ** 5
Observation: Answer: 1104080.8032
Thought:最初の5年間での人口の増加が計算できました。5年後の人口は約1104080人です。次に、次の5年間での人口の減少を計算します。

Action: Calculator
Action Input: 1104080 * (1 - 0.03) ** 5
Observation: Answer: 948111.0630948559
Thought:次の5年間での人口の減少が計算できました。10年後の人口は約948111人です。

Final Answer: 948111人

> Finished chain.


'948111人'

ChainsのTool化

In [12]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain import LLMChain

system_template = """
以下の3つを、それぞれ50文字程度の簡単な紹介付きで答えてください。

おすすめの観光スポット:
1. 兼六園
2. 金沢21世紀美術館
3. 近江町市場

出力:
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{input}")
])

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

chain = LLMChain(prompt=prompt, llm=llm)

/tmp/ipykernel_1426/1076644662.py:22: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(prompt=prompt, llm=llm)


In [13]:
from langchain.tools import Tool

answer_tool = Tool.from_function(
    func=chain.run,
    name="石川県のおすすめの観光スポット",
    description="石川県のおすすめの観光スポットについて聞かれたら答える"
)

In [14]:
from langchain.agents import AgentType, initialize_agent

tools = [answer_tool]

agent_executor = initialize_agent(
    llm=llm,
    tools=tools,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent_executor.run("石川県のおすすめの観光スポットはどこですか？")



> Entering new AgentExecutor chain...
I should find information about recommended tourist spots in Ishikawa Prefecture.  
Action: 石川県のおすすめの観光スポット  
Action Input: None  
Observation: 1. **兼六園**  
日本三名園の一つで、美しい庭園と四季折々の風景が楽しめます。

2. **金沢21世紀美術館**  
現代アートを中心に展示する美術館で、ユニークな建築も魅力です。

3. **近江町市場**  
新鮮な海産物や地元の特産品が揃う市場で、食文化を満喫できます。
Thought:I now have several recommended tourist spots in Ishikawa Prefecture.  
Final Answer: 石川県のおすすめの観光スポットは以下の通りです：
1. **兼六園** - 日本三名園の一つで、美しい庭園と四季折々の風景が楽しめます。
2. **金沢21世紀美術館** - 現代アートを中心に展示する美術館で、ユニークな建築も魅力です。
3. **近江町市場** - 新鮮な海産物や地元の特産品が揃う市場で、食文化を満喫できます。

> Finished chain.


'石川県のおすすめの観光スポットは以下の通りです：\n1. **兼六園** - 日本三名園の一つで、美しい庭園と四季折々の風景が楽しめます。\n2. **金沢21世紀美術館** - 現代アートを中心に展示する美術館で、ユニークな建築も魅力です。\n3. **近江町市場** - 新鮮な海産物や地元の特産品が揃う市場で、食文化を満喫できます。'

In [16]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.chat_models import ChatOpenAI
import wikipedia

wikipedia.set_user_agent("langchain-app/0.1 (test@example.com)")
wikipedia.set_lang("ja")

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

tools = load_tools(["wikipedia", "llm-math"], llm=llm)

agent_executor = initialize_agent(
    llm=llm,
    tools=tools,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [17]:
agent_executor.run("320の15%は何ですか？")



> Entering new AgentExecutor chain...
320の15%を計算する必要があります。これは320に0.15を掛けることと同じです。計算を行います。  
Action: Calculator  
Action Input: 320 * 0.15  
Observation: Answer: 48.0
Thought:I now know the final answer.  
Final Answer: 320の15%は48です。

> Finished chain.


'320の15%は48です。'

自作Tools

In [29]:
#OpenAI のチャットモデル
from langchain.chat_models import ChatOpenAI
#チャット用のプロンプト（指示文）テンプレートを作成
from langchain.prompts import ChatPromptTemplate
#プロンプトと LLM（言語モデル）を1つに連結して実行する
from langchain import LLMChain

#大規模言語モデル（LLM）からの応答（テキスト）を取得する処理(引数はparam(将来的に追加の引数やデータを入れる)とsystem_template(システム指示 ))
#①使用する言語モデル
#②モデルに渡すプロンプト（指示のテンプレート）
def result_chain(param,system_template):
  llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0.5)

  prompt = ChatPromptTemplate.from_messages([
    ("system",system_template),
    ("human","{input}")
  ])

#「プロンプト（指示文）」と「LLM（大規模言語モデル）」を1つの実行可能な処理単位（チェーン）に結合する
  chain = LLMChain(llm=llm,prompt=prompt)

#処理チェーン（Chain）を実行して結果を取得する
  result = chain.run(param)
  return result


In [30]:
from langchain.tools import Tool

def get_childcare_stress_advice(param):
    system_template = """
    あなたは親の育児ストレスを軽減するための専門家です。
    育児疲れやストレス管理に関する実践的なアドバイスを提供します。
    親自身の心身の健康を保つための方法を教えます。
    """

    result = result_chain(param, system_template)
    return result

childcare_stress_advice_tool = Tool.from_function(
    func=get_childcare_stress_advice,
    name="親の育児ストレス軽減に詳しい専門家",
    description="親の育児疲れやストレス管理に関する実践的なアドバイスを提供する"
)

In [31]:
def get_childcare_nutrition_advice(param):
    system_template = """
    あなたは子どもの栄養に詳しいアドバイザーです。
    子どもの健康な発育を支える食事や栄養バランスについてアドバイスを提供します。
    食事の習慣や偏食に関する質問にも丁寧に答えます。
    """

    result = result_chain(param, system_template)
    return result

childcare_nutrition_advice_tool = Tool.from_function(
    func=get_childcare_nutrition_advice,
    name="子どもの栄養に詳しい専門家",
    description="子どもの健康な発育を支える食事や栄養バランスについてアドバイスを提供する"
)

In [32]:
def get_childcare_sleep_advice(param):
    system_template = """
    あなたは子どもの睡眠習慣に詳しい専門家です。
    子どもの夜泣きや睡眠不足に関する解決策を提供し、健全な睡眠を促すためのアドバイスを行います。
    親が子どもの睡眠問題に対処できるようサポートします。
    """

    result = result_chain(param, system_template)
    return result

childcare_sleep_advice_tool = Tool.from_function(
    func=get_childcare_sleep_advice,
    name="子どもの睡眠習慣に詳しい専門家",
    description="子どもの夜泣きや睡眠不足に関する解決策を提供し、健全な睡眠を促すためのアドバイスを行う"
)

In [33]:
def get_childcare_balance_advice(param):
    system_template = """
    あなたは働く親のための育児と仕事の両立に詳しいアドバイザーです。
    仕事と育児のバランスを保つための実践的なアドバイスを提供し、時間管理や家族とのコミュニケーションをサポートします。
    """

    result = result_chain(param, system_template)
    return result

childcare_balance_advice_tool = Tool.from_function(
    func=get_childcare_balance_advice,
    name="働く親のための育児と仕事の両立に詳しい専門家",
    description="仕事と育児のバランスを保つための実践的なアドバイスを提供する"
)

In [34]:
#LangChain（Python向けLLM開発フレームワーク）でLLMにツールを使わせて自律的にタスクを解決させる仕組み（Agent）を構築する
from langchain.agents import AgentType,initialize_agent,load_tools

tools = [
    childcare_stress_advice_tool,
    childcare_nutrition_advice_tool,
    childcare_sleep_advice_tool,
    childcare_balance_advice_tool
]

agent_executor = initialize_agent(
   llm=llm,
   tools=tools,
   agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
   verbose=True
)

In [35]:
agent_executor.run("子どものイヤイヤ期で疲れています。ストレスを軽減する方法を教えてください。")



> Entering new AgentExecutor chain...
子どものイヤイヤ期は親にとって非常にストレスがかかる時期です。ストレスを軽減するためのアドバイスを専門家に求める必要があります。
Action: 親の育児ストレス軽減に詳しい専門家
Action Input: 子どものイヤイヤ期で疲れている親へのストレス軽減方法
Observation: 子どものイヤイヤ期は、親にとって非常にストレスフルな時期です。以下に、親がそのストレスを軽減するための実践的な方法をいくつかご紹介します。

### 1. **感情を受け入れる**
   - イヤイヤ期は自然な成長過程であることを理解し、子どもの感情を受け入れましょう。子どもが自分の意思を表現する大切な時期です。

### 2. **選択肢を与える**
   - 子どもに選択肢を与えることで、自己主張を尊重しつつ、親の意向も反映させることができます。例えば、「赤いシャツと青いシャツ、どっちを着たい？」と聞くと、子どもは自分の意見が尊重されていると感じます。

### 3. **ルーチンを作る**
   - 日常生活において、一定のルーチンを作ることで、子どもは安心感を持ち、イヤイヤを減らすことができます。食事やお風呂の時間を決めておくと良いでしょう。

### 4. **自分の時間を持つ**
   - 忙しい育児の中でも、自分自身の時間を確保することが大切です。短時間でもリラックスできる時間を設け、趣味や運動を楽しむことでストレスを発散しましょう。

### 5. **サポートを求める**
   - 友人や家族にサポートをお願いすることも大切です。育児を一緒に行ったり、話を聞いてもらったりすることで、気持ちが軽くなることがあります。

### 6. **呼吸法やマインドフルネス**
   - 深呼吸や簡単なマインドフルネスの練習を取り入れることで、ストレスを軽減できます。子どもが寝ている間に、自分自身のために数分間静かな時間を持つことも効果的です。

### 7. **ユーモアを持つ**
   - イヤイヤ期の出来事をユーモラスに捉えることで、ストレスを軽減できます。例えば、子どもがイヤイヤをしている時に、面白い声で反応してみると、場の雰囲気が和らぎます。

### 8. **感謝の気持ちを持つ**
 

'子どものイヤイヤ期で疲れている親へのストレス軽減方法には、感情を受け入れる、選択肢を与える、ルーチンを作る、自分の時間を持つ、サポートを求める、呼吸法やマインドフルネス、ユーモアを持つ、感謝の気持ちを持つことが効果的です。これらを試し、自分に合った方法を見つけてください。'

In [36]:
agent_executor.run("2歳の子どもに適したおやつの選び方を教えてください。")



> Entering new AgentExecutor chain...
おやつの選び方についての情報が必要です。子どもの栄養に詳しい専門家に相談するのが適切です。  
Action: 子どもの栄養に詳しい専門家  
Action Input: 2歳の子どもに適したおやつの選び方  
Observation: 2歳の子どもに適したおやつを選ぶ際には、以下のポイントを考慮すると良いでしょう。

1. **栄養バランス**: おやつは、栄養素を補う機会でもあります。果物、野菜、全粒穀物、乳製品、ナッツなど、栄養価の高い食材を選びましょう。

2. **食材の安全性**: 小さな子どもは喉に詰まらせやすいので、硬いものや大きな塊は避けるべきです。果物は小さく切ったり、柔らかいものを選んだりすると良いでしょう。

3. **添加物の確認**: 加工食品やお菓子には添加物が含まれていることが多いので、できるだけ自然な食材を使ったおやつを選ぶことが大切です。

4. **甘さの調整**: 砂糖を多く含むおやつは控えめにし、果物の自然な甘さを活用するのが理想的です。

5. **食事とのバランス**: おやつは食事の補完として位置づけ、食事の時間を妨げないように、適切なタイミングで与えることが重要です。

### おすすめのおやつ例
- **果物**: バナナ、りんご（薄切り）、いちご、みかんなど
- **野菜スティック**: にんじんやきゅうりを細く切って、ディップ（ヨーグルトなど）と一緒に
- **ヨーグルト**: プレーンヨーグルトに果物を混ぜると栄養価がアップ
- **全粒パンやクラッカー**: 小さくカットして、チーズやアボカドをトッピング
- **オートミールクッキー**: 自家製のもので、甘さを控えめにする

子どもの好みや食べやすさを考慮しながら、楽しく健康的なおやつを取り入れていきましょう。
Thought:I now know the final answer.  
Final Answer: 2歳の子どもに適したおやつを選ぶ際には、栄養バランスを考え、食材の安全性を確認し、添加物を避け、甘さを調整し、食事とのバランスを保つことが重要です。おすすめのおやつには、果物（バナナ、りんご、いちごなど）、野菜スティック、ヨーグルト、全粒パンやクラッカー、オートミ

'2歳の子どもに適したおやつを選ぶ際には、栄養バランスを考え、食材の安全性を確認し、添加物を避け、甘さを調整し、食事とのバランスを保つことが重要です。おすすめのおやつには、果物（バナナ、りんご、いちごなど）、野菜スティック、ヨーグルト、全粒パンやクラッカー、オートミールクッキーなどがあります。'

In [37]:
agent_executor.run("昼寝の時間をどのように調整すれば夜の睡眠を妨げませんか？")



> Entering new AgentExecutor chain...
昼寝の時間を調整することで、夜の睡眠に影響を与えないようにする方法について考える必要があります。子どもの睡眠習慣に詳しい専門家に相談するのが適切です。  
Action: 子どもの睡眠習慣に詳しい専門家  
Action Input: 昼寝の時間を調整する方法についてのアドバイスを求める  
Observation: 昼寝の時間を調整することは、子どもの夜の睡眠の質を向上させるために非常に重要です。以下のアドバイスを参考にして、昼寝の時間を適切に調整してみてください。

### 1. 昼寝の時間を決める
- **年齢に応じた昼寝の必要性**: 幼児は一般的に昼寝が必要ですが、年齢が上がるにつれて昼寝の必要性が減ります。例えば、1歳児は1日に1〜2回の昼寝が必要ですが、3歳以上になると1回の昼寝が一般的です。
- **昼寝の時間**: 昼寝は通常、30分から2時間程度が理想です。長すぎる昼寝は夜の睡眠に影響を与えることがあります。

### 2. 昼寝のタイミングを考える
- **午前と午後のバランス**: 昼寝は、午前中の活動の後や午後の早い時間帯に行うのが理想です。午後遅くに昼寝をすると、夜の睡眠に影響を及ぼすことがあります。
- **ルーチンを作る**: 昼寝の時間を毎日同じにすることで、子どもがリズムを覚えやすくなります。

### 3. 環境を整える
- **静かな場所を提供**: 昼寝をする場所は静かで暗い環境が理想です。カーテンで光を遮ったり、静かな音楽を流したりすることが効果的です。
- **快適な温度**: 部屋の温度も重要です。快適な温度に保つことで、子どもがよりリラックスしやすくなります。

### 4. 昼寝後の過ごし方
- **スムーズな移行**: 昼寝から目覚めた後は、すぐに活発な活動を始めるのではなく、少し時間をかけて落ち着く時間を設けると良いでしょう。
- **おやつや軽い活動**: 昼寝後に軽いおやつを与えたり、静かな遊びをすることで、子どもがリフレッシュしやすくなります。

### 5. 反応を観察する
- **子どもの反応を確認**: 昼寝の時間や長さを調整した後は、子どもがどのように反応するかを観察しましょう。夜の睡眠が改善されれば、調整が成功した証

'昼寝の時間を調整するためには、年齢に応じた昼寝の必要性を考慮し、昼寝の時間を30分から2時間程度に設定し、午後遅くではなく午前または午後の早い時間に行うことが重要です。また、環境を整え、昼寝後の過ごし方にも気を配り、子どもの反応を観察することで、夜の睡眠に影響を与えないように調整していくことができます。'

In [38]:
agent_executor.run("忙しい平日に効率よく育児と仕事を両立する方法を教えてください。")



> Entering new AgentExecutor chain...
忙しい平日に育児と仕事を両立するためには、実践的なアドバイスが必要です。働く親のための育児と仕事の両立についての専門家に相談するのが良いでしょう。
Action: 働く親のための育児と仕事の両立に詳しい専門家
Action Input: 忙しい平日に効率よく育児と仕事を両立する方法
Observation: 忙しい平日に育児と仕事を効率よく両立するための方法はいくつかあります。以下に実践的なアドバイスをまとめました。

### 1. **時間管理の工夫**
- **時間割を作成**: 1週間のスケジュールを立て、仕事の時間と育児の時間を明確に分けます。家事の時間も考慮に入れて、計画的に行動しましょう。
- **優先順位をつける**: 仕事や育児のタスクに優先順位をつけ、最も重要なことから取り組むようにします。

### 2. **朝のルーチンを整える**
- **前日の準備**: 洋服やお弁当、必要な書類などを前日に準備しておくことで、朝のバタバタを減らしましょう。
- **早起き**: 家族が起きる前に起きて、自分の時間を持つことで、心の余裕が生まれます。

### 3. **効率的な仕事環境の構築**
- **働きやすい場所**: 在宅勤務の場合は、集中できるスペースを確保し、作業環境を整えます。
- **デジタルツールの活用**: タスク管理アプリやカレンダーを使って、仕事の進捗を可視化しやすくします。

### 4. **家族とのコミュニケーション**
- **定期的なミーティング**: 家族全員が集まる時間を設け、今週の予定や困っていることを話し合います。
- **サポートを求める**: パートナーや家族に助けを求めることをためらわず、協力体制を築くことが大切です。

### 5. **柔軟性を持つ**
- **計画の見直し**: 予定通りに進まないこともありますので、柔軟に対応できるよう心の準備をしておきます。
- **休息を大切に**: 自分自身のリフレッシュタイムを確保することも忘れずに。ストレスを軽減するために、短い休憩を取ることが重要です。

### 6. **育児の効率化**
- **一緒にできること**: 家事や育児を子どもと一緒に行うことで、コミュニケーションを

'忙しい平日に育児と仕事を効率よく両立するためには、時間管理の工夫、朝のルーチンを整える、効率的な仕事環境の構築、家族とのコミュニケーション、柔軟性を持つこと、育児の効率化を図ることが重要です。具体的には、時間割を作成し優先順位をつける、前日に準備をし早起きする、集中できる作業環境を整える、定期的に家族で話し合う、計画を見直しながら柔軟に対応する、そして子どもと一緒にできることを増やすなどの方法があります。これらを実践することで、育児と仕事の両立がスムーズになります。'

In [39]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

tools = load_tools(["llm-math"], llm=llm)

agent_executor = initialize_agent(
    llm=llm,
    tools=tools,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

agent_executor.run("健康に良い習慣を教えて")



> Entering new AgentExecutor chain...
健康に良い習慣について考える必要があります。一般的に、健康に良い習慣には、バランスの取れた食事、定期的な運動、十分な睡眠、ストレス管理、そして水分補給が含まれます。これらの習慣を実践することで、身体的および精神的な健康を促進できます。

Final Answer: 健康に良い習慣には、バランスの取れた食事、定期的な運動、十分な睡眠、ストレス管理、水分補給が含まれます。

> Finished chain.


'健康に良い習慣には、バランスの取れた食事、定期的な運動、十分な睡眠、ストレス管理、水分補給が含まれます。'

In [40]:
from langchain.tools import Tool
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain import LLMChain

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

system_template = "敬語を使わず、友達と喋るノリでリラックスしたトーンでユーザーと会話してください。"
prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{input}")
])

chain = LLMChain(prompt=prompt, llm=llm)

In [41]:
general_answer_tool = Tool.from_function(
    func=chain.run,
    name="数学問題以外の一般的な入力に対しての回答",
    description="数学問題以外の一般的な入力に対して回答を行う"
)

In [42]:
from langchain.agents import AgentType, initialize_agent, load_tools

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

tools = load_tools(["llm-math"], llm=llm)
tools.append(general_answer_tool)

agent_executor = initialize_agent(
    llm=llm,
    tools=tools,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent_executor.run("健康に良い習慣を教えて")



> Entering new AgentExecutor chain...
健康に良い習慣について考え、いくつかのポイントを挙げる必要があります。これには、食事、運動、睡眠、ストレス管理などが含まれます。  
Action: 数学問題以外の一般的な入力に対しての回答  
Action Input: 健康に良い習慣を教えてください。  
Observation: もちろん！健康に良い習慣、いくつか紹介するね。

1. **運動**: 毎日少しでも体を動かすのが大事。散歩とかストレッチでもOK！
2. **水分補給**: 水をしっかり飲むのが大切。ジュースじゃなくて水ね。
3. **バランスの良い食事**: 野菜や果物をたくさん摂るようにするといいよ。色んな食材を食べるのがポイント。
4. **睡眠**: しっかり寝ることも大事。質の良い睡眠を心がけてね。
5. **ストレス管理**: 趣味とかリラックスできる時間を作るのも大切だよ。

こんな感じかな！何か気になることあったら聞いてね。
Thought:私は健康に良い習慣についての情報を得ました。  
Final Answer: 健康に良い習慣には、運動（毎日少しでも体を動かす）、水分補給（十分な水を飲む）、バランスの良い食事（野菜や果物を多く摂る）、睡眠（質の良い睡眠を心がける）、ストレス管理（趣味やリラックス時間を持つ）が含まれます。

> Finished chain.


'健康に良い習慣には、運動（毎日少しでも体を動かす）、水分補給（十分な水を飲む）、バランスの良い食事（野菜や果物を多く摂る）、睡眠（質の良い睡眠を心がける）、ストレス管理（趣味やリラックス時間を持つ）が含まれます。'